In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import json
import time

root = Path("/kaggle/working")
repo = root / "rnn-hg-upstream"
logs = root / "rnn-hg-audit" / "commit1"
logs.mkdir(parents=True, exist_ok=True)

url = "https://github.com/RuiMao1988/Sequential-Metaphor-Identification.git"
commit = "eb924b57415f9d92d3b97df8f14881d5b200ff78"


def run(args, cwd=None, required=True):
    started = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()
    result = subprocess.run(
        args, cwd=cwd, capture_output=True, text=True
    )

    record = {
        "at": started,
        "command": args,
        "cwd": str(cwd) if cwd else None,
        "seconds": round(time.perf_counter() - t0, 3),
        "exit_code": result.returncode,
        "output": result.stdout + result.stderr
    }

    with (logs / "commands.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("$", " ".join(args))
    print(record["output"])

    if required:
        result.check_returncode()

    return result

# Lấy mã nguồn gốc
if not repo.exists():
    run(["git", "clone", "--no-checkout", url, str(repo)])
    run(["git", "checkout", "--detach", commit], cwd=repo)

# Kiểm tra commit gốc
actual = run(["git", "rev-parse", "HEAD"], cwd=repo).stdout.strip()
assert actual == commit, "Repository khác commit đã chọn."

diff = run(["git", "diff", "HEAD", "--"], cwd=repo).stdout
assert not diff, "File tác giả đã bị sửa."

(logs / "source.json").write_text(
    json.dumps({"repository": url, "commit": actual}, indent=2),
    encoding="utf-8"
)

print("SOURCE_OK")

$ git clone --no-checkout https://github.com/RuiMao1988/Sequential-Metaphor-Identification.git /kaggle/working/rnn-hg-upstream
Cloning into '/kaggle/working/rnn-hg-upstream'...

$ git checkout --detach eb924b57415f9d92d3b97df8f14881d5b200ff78
HEAD is now at eb924b5 Delete TroFi-X_formatted_svo.csv

$ git rev-parse HEAD
eb924b57415f9d92d3b97df8f14881d5b200ff78

$ git diff HEAD --

SOURCE_OK


In [5]:
import ast

# Đọc yêu cầu tác giả
readme = (repo / "README.md").read_text(encoding="utf-8")
requirements = (repo / "requirements.txt").read_text(encoding="utf-8")

print(readme)
print("REQUIREMENTS:\n", requirements)

# Trích cấu hình gốc
source = (repo / "rnn_hg/main_vua.py").read_text(encoding="utf-8")
tree = ast.parse(source)

names = {
    "r", "using_GPU", "batch_size",
    "num_epochs", "additional_epochs"
}
config = {}

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in names:
                config[target.id] = ast.literal_eval(node.value)

# Đọc kiến trúc gốc
for node in ast.walk(tree):
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        if node.func.id == "RNNSequenceModel":
            config["model"] = {
                item.arg: ast.literal_eval(item.value)
                for item in node.keywords
            }

# Đọc thiết lập train
for line in source.splitlines():
    if any(text in line for text in (
        "optim.SGD(",
        "class_weights =",
        "loss_criterion =",
        "cudnn.deterministic",
        "glove_embeddings =",
        "h5py.File("
    )):
        print(line.strip())

print(json.dumps(config, indent=2))

(logs / "original_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)
(logs / "original_requirements.txt").write_text(
    requirements, encoding="utf-8"
)

# Sequential-Metaphor-Identification
Code for the paper "End-to-End Sequential Metaphor Identification Inspired by Linguistic Theories"

https://www.aclweb.org/anthology/papers/P/P19/P19-1378/

## Embeddings
Download glove.840B.300d.zip (https://nlp.stanford.edu/projects/glove/) into glove folder.

Please email terry_mao0513@hotmail.com for ELMo embeddings, putting them into elmo folder.

(The used dataset and ELMo embeddings were cleaned and pre-trained by https://github.com/gao-g/metaphor-in-context.)

## Environment
python 3.6

pytorch 0.4.1
```
pip install -r requirements.txt
```
## Run
Go to the model folder (rnn_hg or rnn_mhca), then run for

1. VU Amsterdam Metaphor Corpus (VUA ALL POS and VUA VERB in the paper)
```
python main_vua.py
```
2. Mohammad dataset
```
python main_mohx.py
```
3. TroFi dataset
```
python main_trofi.py
```
## Citation
```
@inproceedings{mao2019metaphor,
  title={End-to-End Sequential Metaphor Identification Inspired by Linguistic Theories},
  author={Mao

201

In [6]:
import os
import sys
import shutil
import platform

# Kiểm tra máy chạy
hardware = {
    "at": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "ram_bytes": os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
    "disk_free_bytes": shutil.disk_usage(root).free
}

print(json.dumps(hardware, indent=2))

# Kiểm tra GPU hiện tại
if shutil.which("nvidia-smi"):
    gpu = run(["nvidia-smi"], required=False)
    hardware["nvidia_smi"] = gpu.stdout + gpu.stderr
else:
    hardware["nvidia_smi"] = "Không tìm thấy nvidia-smi."
    print(hardware["nvidia_smi"])

# Kiểm tra PyTorch notebook
probe = """
import torch
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("CUDA available:", torch.cuda.is_available())
"""

torch_check = run(
    [sys.executable, "-c", probe],
    required=False
)

hardware["torch_check_exit_code"] = torch_check.returncode
hardware["torch_output"] = torch_check.stdout + torch_check.stderr

(logs / "hardware.json").write_text(
    json.dumps(hardware, indent=2),
    encoding="utf-8"
)

{
  "at": "2026-11-16T07:25:39.322694+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "python_executable": "/usr/bin/python3",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "cpu_count": 4,
  "ram_bytes": 33658159104,
  "disk_free_bytes": 20873940992
}
$ nvidia-smi
Mon Nov 16 07:25:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4      

2519

In [7]:
import csv
import hashlib
from itertools import combinations

# Kiểm tra toàn dữ liệu
stats = {}
ids = {}

for split in ("train", "val", "test"):
    path = repo / "data/VUAsequence" / f"VUA_seq_formatted_{split}.csv"

    sentence_ids = set()
    sentences = tokens = positives = duplicates = 0

    with path.open(encoding="latin-1", newline="") as f:
        for row in csv.DictReader(f):
            words = row["sentence"].split()
            labels = ast.literal_eval(row["label_seq"])
            pos = ast.literal_eval(row["pos_seq"])

            assert len(words) == len(labels) == len(pos), (
                split, row["txt_id"], row["sen_ix"]
            )
            assert set(labels) <= {0, 1}, (
                split, row["txt_id"], row["sen_ix"]
            )

            key = (row["txt_id"], row["sen_ix"])
            duplicates += int(key in sentence_ids)
            sentence_ids.add(key)

            sentences += 1
            tokens += len(words)
            positives += sum(labels)

    ids[split] = sentence_ids
    stats[split] = {
        "sentences": sentences,
        "tokens": tokens,
        "metaphor_tokens": positives,
        "literal_tokens": tokens - positives,
        "metaphor_percent": round(100 * positives / tokens, 2),
        "duplicate_sentence_ids": duplicates,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
    }

# Kiểm tra giao ID
overlap = {
    f"{a}__{b}": len(ids[a] & ids[b])
    for a, b in combinations(ids, 2)
}

report = {
    "splits": stats,
    "sentence_id_overlap": overlap
}

(logs / "data_check.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print(json.dumps(report, indent=2))

# Đối chiếu dữ liệu
expected = {
    "train": (6323, 116622, 13051),
    "val": (1550, 38628, 4487),
    "test": (2694, 50175, 6243)
}

for split, counts in expected.items():
    found = stats[split]

    assert (
        found["sentences"],
        found["tokens"],
        found["metaphor_tokens"]
    ) == counts

    assert found["duplicate_sentence_ids"] == 0

assert all(count == 0 for count in overlap.values())

print("DATA_OK")
print("Nhật ký:", logs)

{
  "splits": {
    "train": {
      "sentences": 6323,
      "tokens": 116622,
      "metaphor_tokens": 13051,
      "literal_tokens": 103571,
      "metaphor_percent": 11.19,
      "duplicate_sentence_ids": 0,
      "sha256": "1c2db64a543f999626b765b0cf1f00d054a820b0ea005ccb536aaf1ff33f9b43"
    },
    "val": {
      "sentences": 1550,
      "tokens": 38628,
      "metaphor_tokens": 4487,
      "literal_tokens": 34141,
      "metaphor_percent": 11.62,
      "duplicate_sentence_ids": 0,
      "sha256": "66dc24219c2ba21d53d70ef0b0eac895f52cf18524238674cff4448e23feab3c"
    },
    "test": {
      "sentences": 2694,
      "tokens": 50175,
      "metaphor_tokens": 6243,
      "literal_tokens": 43932,
      "metaphor_percent": 12.44,
      "duplicate_sentence_ids": 0,
      "sha256": "33256891fb84cdda539def4781409762de6af645a415fbd13c7fcc08e5ee6135"
    }
  },
  "sentence_id_overlap": {
    "train__val": 0,
    "train__test": 0,
    "val__test": 0
  }
}
DATA_OK
Nhật ký: /kaggle/working/rnn

## Kết luận kiểm tra

- Mã nguồn đúng commit
- Train/dev/test có 6323/1550/2694 câu.
- Token, nhãn và POS khớp độ dài.
- Không trùng ID câu trong và giữa các tập.
- Kaggle: Python 3.13.16, PyTorch 2.11.0+cu130.
- GPU: hai Tesla T4, CUDA khả dụng.
- Tác giả yêu cầu Python 3.6, PyTorch 0.4.1.
